# Treino EDSR-Baseline (Enhanced Deep Residual Networks)

**Referência:** Lim et al., 2017 - "Enhanced Deep Residual Networks for Single Image Super-Resolution"

Este notebook contém apenas o treino do modelo EDSR-Baseline (16 blocos residuais, 64 filtros, sem BatchNorm). Para análise exploratória dos dados e comparação com outros modelos, veja `00_guia_analise_comparacao.ipynb`.

## Setup e Configurações

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

sys.path.append(".")

from datasets import SRDataset, SRBenchmarkDataset, SREvalDataset
from models import EDSRBaseline
from metrics import calc_psnr, calc_ssim
from train import fit, evaluate

In [ ]:
# Configurar seeds para reprodutibilidade
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seed fixada: {SEED}")

In [ ]:
# Detectar dispositivo
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}")

In [ ]:
# Criar diretório de outputs
student_tag = "EDSR_2026-07-05"  # Altere conforme necessário
output_dir = Path("outputs") / student_tag
output_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(output_dir / "tensorboard"))

print(f"Output dir: {output_dir}")
print(f"TensorBoard: tensorboard --logdir {output_dir / 'tensorboard'}")

## Carregamento de Dados

In [ ]:
# Configurações de dataset
PATCH_SIZE = 192  # EDSR usa patches maiores
SCALE = 4
BATCH_SIZE = 16

# Verificar se dados existem
DATA_DIR = Path("datasets")
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Pasta '{DATA_DIR}' não encontrada. Baixe os datasets (DIV2K, Set5, etc.) antes de continuar.")

# Carregar datasets
train_dataset = SRDataset("datasets/DIV2K_train_HR", patch_size=PATCH_SIZE, scale=SCALE)
val_dataset = SRBenchmarkDataset("datasets/Set5/image_SRF_4", scale=SCALE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

print(f"Dataset de treino: {len(train_dataset)} patches")
print(f"Dataset de validação: {len(val_dataset)} imagens")
print(f"Batch size: {BATCH_SIZE} | Patch size: {PATCH_SIZE}x{PATCH_SIZE} | Scale: {SCALE}x")

In [ ]:
# Visualizar amostra do batch de treino
sample_loader = DataLoader(train_dataset, batch_size=6, shuffle=True)
lr_batch, hr_batch = next(iter(sample_loader))

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i in range(6):
    axes[0, i].imshow(lr_batch[i].permute(1, 2, 0))
    axes[0, i].set_title(f"LR {tuple(lr_batch[i].shape[1:])}")
    axes[0, i].axis("off")

    axes[1, i].imshow(hr_batch[i].permute(1, 2, 0))
    axes[1, i].set_title(f"HR {tuple(hr_batch[i].shape[1:])}")
    axes[1, i].axis("off")

plt.suptitle("Amostra de patches do batch de treino (LR em cima, HR embaixo)")
plt.tight_layout()
plt.savefig(output_dir / "sample_batch.png", dpi=150)
plt.show()

## Treinamento

In [ ]:
# Inicializar modelo
model = EDSRBaseline(num_features=64, num_blocks=16, scale=SCALE).to(device)

# Configuração de treino
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.L1Loss()  # EDSR usa L1Loss em vez de MSELoss
n_epochs = 100

print(f"Modelo: {model.__class__.__name__}")
print(f"Parâmetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"Configuração: num_features=64, num_blocks=16, scale={SCALE}")
print(f"Optimizer: Adam (lr=1e-4)")
print(f"Loss: L1Loss")
print(f"Epochs: {n_epochs}")

In [ ]:
# Treino
edsr_history = fit(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=n_epochs, writer=writer, tag="edsr", upscale_input=False
)

print("\n✅ Treinamento concluído!")

## Avaliação e Salvamento

In [ ]:
# Avaliar em dataset DIV2K_valid_HR
eval_dataset = SREvalDataset("datasets/DIV2K_valid_HR", scale=SCALE)
eval_loader = DataLoader(eval_dataset, batch_size=1, shuffle=False)

val_metrics = evaluate(model, eval_loader, device, upscale_input=False)

print(f"\nResultados em DIV2K_valid:")
print(f"  PSNR (modelo): {val_metrics['psnr_model']:.2f} dB")
print(f"  PSNR (bicubic baseline): {val_metrics['psnr_bicubic']:.2f} dB")
print(f"  SSIM: {val_metrics['ssim_model']:.4f}")
print(f"  Ganho em relação ao bicubic: {val_metrics['psnr_model'] - val_metrics['psnr_bicubic']:+.2f} dB")

In [ ]:
# Salvar modelo
model_path = output_dir / "edsr_final.pt"
torch.save(model.state_dict(), model_path)

print(f"✅ Modelo salvo em: {model_path}")

In [ ]:
# Visualizar resultados
def tensor_to_img(t):
    return t.detach().squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1)

model.eval()
n_samples = 5

fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))

with torch.no_grad():
    for i, (lr_img, hr_img) in enumerate(eval_loader):
        if i >= n_samples:
            break

        lr_img, hr_img = lr_img.to(device), hr_img.to(device)
        lr_up = F.interpolate(lr_img, scale_factor=SCALE, mode="bicubic", align_corners=False)
        sr = model(lr_img).clamp(0, 1)

        psnr_bic = calc_psnr(lr_up, hr_img)
        psnr_sr = calc_psnr(sr, hr_img)

        axes[i, 0].imshow(tensor_to_img(hr_img))
        axes[i, 0].set_title("HR (original)")

        axes[i, 1].imshow(tensor_to_img(lr_up))
        axes[i, 1].set_title(f"Bicubic\nPSNR: {psnr_bic:.2f} dB")

        axes[i, 2].imshow(tensor_to_img(sr))
        axes[i, 2].set_title(f"EDSR\nPSNR: {psnr_sr:.2f} dB")

        for ax in axes[i]:
            ax.axis("off")

plt.tight_layout()
plt.savefig(output_dir / "resultados_edsr.png", dpi=150)
plt.show()